# 08 — Similarity features (Phase 8)

So far one hand-made number decided everything: the average of name Jaccard and address Jaccard.
Now we compute **many** signals ("features") for each candidate pair. In Phase 9 a model learns how to combine them.

A **feature** is one number describing a (S1 entity, candidate) pair, e.g. "how similar are the names?".
Every feature is computed from the records' **text only** (never the labels), so the same code runs on the test set.

This notebook:
1. teaches each similarity measure on tiny examples;
2. shows how well each feature *alone* separates true from false pairs (AUC);
3. looks at the features of a few real pairs.

The heavy work (2M candidate pairs) is done by `experiments/phase8_features.py`, using `src/features.py`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from rapidfuzz import fuzz

from src.features import FEATURE_COLUMNS

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 60)

---
## 1. Edit distance (Levenshtein) → `fuzz.ratio`

### Concept
The **Levenshtein distance** is the minimum number of single-character edits (insert, delete, replace)
to turn one string into the other. `fuzz.ratio` turns it into a similarity from 0 to 100
(100 = identical). We divide by 100 → 0..1.

### Why do we need it?
Word Jaccard treats `offie` and `office` as completely different words. Edit distance sees they differ by one letter.

### Limitations
It compares characters **in order**, so reordered words (`office davis family`) score low.

In [2]:
examples = [("davis family office", "davis family offie"),
            ("davis family office", "office davis family"),
            ("maure williams colombier", "maurewilliamscolombier"),
            ("abc restaurant", "xyz traders")]
pd.DataFrame([(a, b, fuzz.ratio(a, b)) for a, b in examples], columns=["A", "B", "fuzz.ratio"])

,A,B,fuzz.ratio
0,davis family office,davis family offie,97.297297
1,davis family office,office davis family,63.157895
2,maure williams colombier,maurewilliamscolombier,95.652174
3,abc restaurant,xyz traders,32.000000


## 2. Order-insensitive variants: `token_sort_ratio`, `token_set_ratio`, `partial_ratio`

| scorer | what it does | good for |
|---|---|---|
| `token_sort_ratio` | sorts the words alphabetically, then `ratio` | reordered words |
| `token_set_ratio` | compares the **common** words with each side's extra words; 100 if one word set contains the other | extra words (`llc`, `center`, `(india)`) |
| `partial_ratio` | best match of the shorter string inside the longer one | one name embedded in a longer name |

In [3]:
pairs = [("orient ventures private limited", "orient limited private ventures"),
         ("american program", "american program inc"),
         ("suresh broadcasters", "pyrabelo suresh broadcasters private limited"),
         ("davis family office", "davis family offie")]
pd.DataFrame([(a, b, fuzz.ratio(a, b), fuzz.token_sort_ratio(a, b), fuzz.token_set_ratio(a, b), fuzz.partial_ratio(a, b))
              for a, b in pairs], columns=["A", "B", "ratio", "token_sort", "token_set", "partial"])

,A,B,ratio,token_sort,token_set,partial
0,orient ventures private limited,orient limited private ventures,64.516129,100.000000,100.000000,67.857143
1,american program,american program inc,88.888889,88.888889,100.000000,100.000000
2,suresh broadcasters,pyrabelo suresh broadcasters private limited,60.317460,60.317460,100.000000,100.000000
3,davis family office,davis family offie,97.297297,97.297297,97.297297,97.142857


**Common mistake:** `token_set_ratio` is 100 whenever one name's words are a subset of the other's —
`family center` vs `family center of hagerstown` → 100. That's why the model needs *several* features, not just one.

## 3. TF-IDF cosine on character 3-grams

### Concept
1. Cut each string into overlapping 3-character pieces: `wayne` → ` wa`, `way`, `ayn`, `yne`, `ne `.
2. **TF** (term frequency) = how often each piece occurs in this string.
3. **IDF** (inverse document frequency, learned from 1M unlabeled pool records) = rare pieces get a big weight, common pieces (`ing`, ` st`) a small one.
4. Each string becomes a vector of TF × IDF weights; **cosine similarity** = how much two vectors point in the same direction (0..1).

### Why do we need it?
It is typo-tolerant (most 3-grams survive a typo) **and** it knows which parts are distinctive (`colombier` counts more than `street`).

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

toy_corpus = ["85 wayne avenue ticonderoga", "12 main street springfield", "44 main street troy",
              "7 main street salem", "85 wanye avenue ticonderoga township"]
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 3)).fit(toy_corpus)
vectors = vectorizer.transform(toy_corpus)
cosine = (vectors @ vectors.T).toarray()
pd.DataFrame(cosine.round(2), index=toy_corpus, columns=range(len(toy_corpus)))

,0,1,2,3,4
85 wayne avenue ticonderoga,1.00,0.00,0.00,0.00,0.63
12 main street springfield,0.00,1.00,0.33,0.33,0.00
44 main street troy,0.00,0.33,1.00,0.43,0.00
7 main street salem,0.00,0.33,0.43,1.00,0.00
85 wanye avenue ticonderoga township,0.63,0.00,0.00,0.00,1.00


`85 wayne avenue ticonderoga` vs `85 wanye avenue ticonderoga township` (typo + extra word) still score high,
while the three `main street` addresses resemble each other only through common pieces.

## 4. Hand-made features from what we learned in Phases 2–5

| feature | meaning | why |
|---|---|---|
| `core_*` | name similarity **after removing legal words** | `abc pvt ltd` vs `abc` should look similar |
| `core_joined_ratio` | same, with spaces removed | website names `maurewilliamscolombier` |
| `legal_same / legal_conflict / legal_one_missing` | legal forms equal / clash (`llc` vs `ltd`) / missing on one side | Phase 5's false merges differed only in the legal form |
| `num_shared, num_jaccard, num_conflict, num_c_subset` | house / plot numbers in common, clashing, all candidate numbers found in S1 | `508` vs `517 stack pole road` false merge |
| `addr_c_empty` | candidate has no address | then address features are *unknown* (NaN), not "different" |
| `name_c_foreign_script` | candidate name in an Indian script | name similarity is meaningless there |
| `block_*` | blocking evidence: idf sum of shared rare keys, rank among the entity's candidates, score relative to the best candidate | "is this the best candidate this entity has?" |
| `*_rel` | similarity minus the best similarity among the entity's candidates | helps decide singletons: if nothing is close, predict nothing |
| `is_source3` | candidate comes from Source 3 | the sources have different noise styles |

---
## 5. How well does each feature separate true from false pairs on its own?

### Concept: AUC
Take one true pair and one false pair at random. **AUC** = probability that the feature ranks the true pair higher.
- 1.0 = perfect separation, 0.5 = useless (coin flip), below 0.5 = works "upside down" (lower value = match).
- `separation` = |AUC − 0.5| × 2, so 1 = perfect and 0 = useless in either direction.

Computed on the **tuning** entities' candidate pairs.

In [5]:
auc = pd.read_csv(PROJECT_ROOT / "experiments" / "phase8_feature_auc.tsv", sep="\t")
auc

,feature,AUC,direction,separation,missing_%,mean_true,mean_false
0,block_rank,0.0355,lower = match,0.9289,0.0,3.782,26.995
1,addr_token_set,0.9509,higher = match,0.9018,1.4,0.940,0.596
2,addr_tfidf,0.9492,higher = match,0.8983,1.4,0.854,0.345
3,block_score,0.9491,higher = match,0.8982,0.0,85.847,25.700
4,block_score_rel,0.9469,higher = match,0.8937,0.0,0.791,0.278
5,addr_jaccard,0.9452,higher = match,0.8903,1.4,0.716,0.224
6,num_jaccard,0.9418,higher = match,0.8835,9.8,0.849,0.090
7,addr_token_sort,0.9258,higher = match,0.8516,1.4,0.875,0.545
8,addr_token_set_rel,0.9197,higher = match,0.8394,0.0,-0.081,-0.395
9,addr_ratio,0.8687,higher = match,0.7374,1.4,0.815,0.535


**Reading the AUC table**
- **Address features are the strongest single signals** (`addr_token_set`, `addr_tfidf` ≈ 0.95), better than any name feature
  (best ≈ 0.84). This confirms Phase 3: names are noisy and shared by look-alikes, addresses are distinctive.
- **Blocking evidence is very informative**: `block_rank` (AUC 0.036, i.e. *lower rank = match*) and `block_score` ≈ 0.95.
  True matches are usually the top few candidates.
- **Numbers**: `num_jaccard` 0.94. And `num_conflict` is 1 for **73%** of false pairs but only **4.7%** of true pairs.
- **Legal conflict** (`llc` vs `ltd`) almost never happens in true pairs (0.2%) but in 14.8% of false pairs.
  Its AUC is modest only because most pairs have no conflict either way — when it fires, it is a strong "no".
- Order-insensitive and typo-tolerant name scores (`name_tfidf`, `token_sort`) beat plain word Jaccard.
- `n_candidates` is useless on its own (almost every entity has 50) — the model may ignore it.
- A feature can be weak alone but useful in combination (e.g. `addr_c_empty` tells the model *when* to trust name features).
  That's what the model in Phase 9 figures out.

In [6]:
features = pd.read_parquet(PROJECT_ROOT / "cache" / "phase8_features.parquet")
tuning = features[features["query_set"] == "tuning"]
print(f"{len(features):,} candidate pairs | tuning {len(tuning):,} | positives in tuning: {tuning['label'].mean():.1%}")
tuning.groupby("label")[["name_token_set", "core_joined_ratio", "addr_token_set", "addr_tfidf", "num_shared",
                         "legal_conflict", "block_score_rel", "block_rank"]].mean().round(3).T.rename(columns={0: "false pairs", 1: "true pairs"})

1,997,854 candidate pairs | tuning 998,797 | positives in tuning: 6.5%


label,false pairs,true pairs
name_token_set,0.519000,0.883
core_joined_ratio,0.465000,0.854
addr_token_set,0.596000,0.940
addr_tfidf,0.345000,0.854
num_shared,0.220000,1.239
legal_conflict,0.148000,0.002
block_score_rel,0.278000,0.791
block_rank,26.995001,3.782


### Real pairs: features of true matches that the old Jaccard rule missed

In [7]:
cols = ["name_s1", "name_c", "addr_s1", "addr_c", "name_token_set", "core_joined_ratio", "addr_tfidf", "num_shared", "block_rank"]
old_jaccard = tuning[["name_jaccard", "addr_jaccard"]].mean(axis=1).fillna(0)
tuning[(tuning["label"] == 1) & (old_jaccard < 0.45)][cols].sample(8, random_state=1)

,name_s1,name_c,addr_s1,addr_c,name_token_set,core_joined_ratio,addr_tfidf,num_shared,block_rank
1706517,btl brothers pvt ltd,shri btl brothers pvt,c 22 shri mavji rathod soc juhu road mumbai maharashtra,c 22 mumbai mh,0.888889,0.846154,0.323787,1.0,17.0
710975,soder chain l l c,soderchain,128 pine street east aurora ny,128 pine street east aurora city ny,0.400000,0.869565,0.958694,1.0,7.0
1671369,creative tech private limited,techprivate,c o v lakshmi 192 ground floor jyothi road kranthi nagar...,kurnool kurnool region 192 ground floor jyothi road kran...,0.450000,0.521739,0.910903,1.0,3.0
38576,united business private limited,ୟୁନାଇଟେଡ୍ ବିଜନେସ୍ ପ୍ରାଇଭେଟ୍ ଲିମିଟେଡ୍,plot no 228 unit iii khordha bhubaneswar kharavela nagar...,plot no 228 unit iii kharavela nagar khordha bhubaneswar...,0.089552,0.000000,0.878179,1.0,3.0
1737797,ig marcom private limited,igmarcom,khasra no 403 jmb farm kanadia road indore madhya pradesh,khasra no 403 jmb farm kanadia road indore mp,0.484848,1.000000,0.852232,1.0,6.0
423886,royal marketing private limited,ରୟାଲ୍ ମାର୍କେଟିଂ ପ୍ରାଇଭେଟ୍ ଲିମିଟେଡ୍,plot no 241 p ambapua housing board house no mig 4 1 ber...,plot no 241 p ambapua housing board ganjam berhampur blo...,0.092308,0.000000,0.931893,3.0,5.0
716147,dynamic builders,dynamic बिल्डर्स,6 lightbridge hiranandani busipark chandivali powai mumb...,6 lightbridge hiranandani mumbai mh,0.608696,0.466667,0.706688,1.0,7.0
342078,byrom capital forefront inc,forefrontcapital,431 corona road tucson az,431 corona road tucson arizona,0.465116,0.486486,0.837781,1.0,3.0


### Real pairs: false candidates with very similar names (the look-alikes)

In [8]:
tuning[(tuning["label"] == 0) & (tuning["name_token_set"] >= 0.95)][cols + ["legal_conflict", "num_conflict"]].sample(8, random_state=1)

,name_s1,name_c,addr_s1,addr_c,name_token_set,core_joined_ratio,addr_tfidf,num_shared,block_rank,legal_conflict,num_conflict
1572639,sai foundation limited,sai foundation private limited,a 202 balaji oasis palaj gandhinagar gandhinagar gandhin...,plot d 762 11 4 6 bell haven chs ltd marol maroshi road ...,1.0,1.000000,0.139515,0.0,18.0,0,1
1839951,cure bike shop pc,cure bike shop,5957 coquillo place marana az,ri cumberland township 47 blanche avenue,1.0,1.000000,0.000000,0.0,42.0,0,1
684175,back alley bike shop,back alley bike shop corporation,or 2400 scenic drive portland,5129 wilcox road whitesboro cdp ny,1.0,1.000000,0.000000,0.0,34.0,0,1
1455955,bontrager college center,south college center bontrager,salisbury 1030 glover road nc,1034 1038 glover road n a salisbury nc,1.0,0.612245,0.776972,0.0,3.0,0,1
1966073,national initiative,national initiative inc,5852 macaw avenue city of el paso tx,in 318b willard drive auburn,1.0,1.000000,0.000000,0.0,26.0,0,1
1444450,alpha food llp,alpha food,c o uttam biswas nadia west bengal nadia at new kalikpur...,survey no 777 umarsadi desaiwad road at village umarsadi...,1.0,1.000000,0.062925,0.0,50.0,0,0
1682767,qunet,qunet,2218 cedar ridge drive joliet il,627 rice circle cookeville tn,1.0,1.000000,0.031947,0.0,28.0,0,1
266894,ram marketing private limited,private ram marketing limited,57 lb road kaba plaza 3rd floor 3b adyar chennai tamil nadu,846 saluja complex g t road laxmipur burdwan i bardhaman...,1.0,1.000000,0.016700,0.0,32.0,0,1


---
# Summary — Phase 8

- **34 features** per candidate pair for 2.0M pairs (tuning + small validation), computed in ~4 minutes
  (`src/features.py`, `experiments/phase8_features.py`, saved to `cache/phase8_features.parquet`).
- 6.5% of candidate pairs are true matches → the model must reject ~15 look-alikes for every true match.
- Strongest single signals: blocking rank/score, address similarity (token set, TF-IDF), shared house numbers.
- Strong "different business" signals: number conflict, legal-form conflict.
- Everything is computed from text only, with TF-IDF fitted on unlabeled pool records → identical code runs on the test set.

**Next (Phase 9):** train models (logistic regression → gradient boosting) on the **tuning** entities' pairs,
evaluate macro F0.5 on the **validation** entities, and compare with the rule baseline (0.706).